In [ ]:
# Install YOLOv8
!pip install ultralytics

# Check if the GPU is working (It should print "True")
import torch
print("GPU is enabled:", torch.cuda.is_available())

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 55.5 MB/s eta 0:00:00
GPU is enabled: True


In [ ]:
!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="qFeAudPyHM3ZdmDyDA0w")
project = rf.workspace("urbansenseanonymization").project("anonymization-enogl")
version = project.version(2)
dataset = version.download("yolov8")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.8/95.8 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 15.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 48.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 38.3 MB/s eta 0:00:00
  Attempting uninstall: opencv-python-headless
    Found existing installation: opencv-python-headless 4.13.0.92
    Uninstalling opencv-python-headless-4.13.0.92:
      Successfully uninstalled opencv-python-headless-4.13.0.92
  Attempting uninstall: idna
    Found existing installation: idna 3.11
    Uninstalling idna-3.11:
      Successfully uninstalled idna-3.11
loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to anonymization-2 in yolov8:: 100%|██████████| 12582/12582 [00:01<00:00, 6525.51it/s]


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [ ]:
from ultralytics import YOLO

# 1. Load the pre-trained Nano model (Transfer Learning)
model = YOLO('yolov8n.pt')

# 2. Train the model on your downloaded dataset
# Note: dataset.location contains the path to the downloaded data.yaml
results = model.train(
    data=f"{dataset.location}/data.yaml",
    epochs=30,        # 30 epochs is usually enough for a prototype
    imgsz=640,        # Resolution (416 is great for mobile)
    batch=16,         # Batch size
    name='blurguard_custom'
)

Ultralytics 8.4.26 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/anonymization-2/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=30, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=blurguard_custom, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=

In [ ]:
from ultralytics import YOLO

# 1. Load YOUR trained model, NOT the internet model
# Replace this string with the path you copied in Step 1
trained_model_path = "/content/best.pt"
model = YOLO(trained_model_path)

# 2. Define the path to your data.yaml
# Replace this string with the path you copied in Step 2
yaml_path = "/content/anonymization-2/data.yaml"

# 3. Export to Android (Int8 CPU Optimized)
print("Starting export... this might take 3-5 minutes.")
exported_path = model.export(
    format="tflite",
    int8=True,
    imgsz=640,        # Must match the size you trained with!
    data=yaml_path    # Required for int8 to work
)

print(f"Success! Model exported to: {exported_path}")

Starting export... this might take 3-5 minutes.
Ultralytics 8.4.26 🚀 Python-3.12.13 torch-2.10.0+cu128 CPU (Intel Xeon CPU @ 2.00GHz)
Model summary (fused): 73 layers, 3,006,038 parameters, 0 gradients, 8.1 GFLOPs

PyTorch: starting from '/content/best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 6, 8400) (6.0 MB)

TensorFlow SavedModel: starting export with tensorflow 2.19.0...
TensorFlow SavedModel: collecting INT8 calibration images from 'data=/content/anonymization-2/data.yaml'
Fast image access ✅ (ping: 0.0±0.0 ms, read: 22.7±9.4 MB/s, size: 50.0 KB)
Scanning /content/anonymization-2/valid/labels.cache... 1256 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1256/1256 154.9Mit/s 0.0s
